# DS 301 Midterm Project: Adult Income Classification

## Introduction
This notebook reproduces a classification workflow inspired by Li-Pang Chen's 2021 paper on US adult income prediction.
The project uses the public UCI Adult dataset, where the task is to predict whether a person's annual income exceeds $50K from census attributes.

## Objective
- Load and inspect a real public classification dataset
- Preprocess mixed numeric and categorical data
- Perform exploratory data analysis
- Train and compare at least two classification models
- Evaluate performance with accuracy, confusion matrix, and classification report


## Research Paper Summary
The selected paper is *Supervised Learning for Binary Classification on US Adult Income* by Li-Pang Chen, published in 2021 in the *Journal of Modeling and Optimization*.
The paper analyzes the Adult income dataset and compares several classifiers, including logistic regression and support vector machine, using metrics such as ROC curves, accuracy, recall, specificity, precision, and F-measure.
For this DS 301 project, we reproduce the classification setting using only class-approved algorithms: Logistic Regression and SVM.


In [ ]:
# Import core libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)


In [ ]:
# Load the Adult dataset directly from the official UCI repository
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

train_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
test_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'

train_df = pd.read_csv(train_url, names=columns, na_values='?', skipinitialspace=True)
test_df = pd.read_csv(test_url, names=columns, na_values='?', skipinitialspace=True, skiprows=1)

df = pd.concat([train_df, test_df], ignore_index=True)
df['income'] = df['income'].str.replace('.', '', regex=False).str.strip()

print('Dataset loaded successfully.')
print('Combined shape:', df.shape)


In [ ]:
# Basic data exploration
print('First five rows:')
display(df.head())

print('Dataset shape:')
print(df.shape)

print('DataFrame info:')
df.info()

print('Target distribution:')
print(df['income'].value_counts())


## Data Preprocessing Plan
- Replace placeholder missing values with proper NaN values
- Check missing-value counts
- Keep valid extreme values but inspect outliers numerically
- One-hot encode categorical features
- Standardize numeric features
- Use pipelines to prevent data leakage


In [ ]:
# Missing values and duplicate checks
print('Missing values by column:')
print(df.isnull().sum().sort_values(ascending=False))

print('\nTotal missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

df['income_binary'] = df['income'].map({'<=50K': 0, '>50K': 1})

print('\nTarget encoding preview:')
print(df[['income', 'income_binary']].head())


In [ ]:
# Summary statistics
print('Summary statistics for all columns:')
display(df.describe(include='all').T)

numeric_cols = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'income_binary']
print('Numeric summary statistics:')
display(df[numeric_cols].describe().T)


In [ ]:
# Plot 1: Target class distribution
income_counts = df['income'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(income_counts.index, income_counts.values, color=['steelblue', 'tomato'])
plt.title('Income Class Distribution')
plt.xlabel('Income Class')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

# Plot 2: Age distribution by income class
plt.figure(figsize=(8, 5))
for label, color in [('<=50K', 'steelblue'), ('>50K', 'tomato')]:
    subset = df[df['income'] == label]['age']
    plt.hist(subset, bins=30, alpha=0.5, label=label, color=color)

plt.title('Age Distribution by Income Class')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# Correlation matrix for numeric features
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Matrix (Numeric Features)')
plt.tight_layout()
plt.show()

display(corr)


## Outlier Check Explanation
This dataset is socioeconomic census data, so some large values such as capital gain and capital loss may be legitimate observations rather than errors.
We will inspect outliers using the IQR rule, report their counts, and keep them unless there is clear evidence of data corruption.


In [ ]:
# Outlier inspection using the IQR rule
numeric_features = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
outlier_summary = {}

for col in numeric_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = count

outlier_df = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['outlier_count'])
display(outlier_df.sort_values('outlier_count', ascending=False))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.boxplot(df['capital_gain'].dropna())
plt.title('Boxplot: capital_gain')

plt.subplot(1, 2, 2)
plt.boxplot(df['hours_per_week'].dropna())
plt.title('Boxplot: hours_per_week')

plt.tight_layout()
plt.show()


In [ ]:
# Prepare feature matrix and target vector
X = df.drop(columns=['income', 'income_binary'])
y = df['income_binary']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)


## Methodology
We implement two class-approved models from the paper context:
- Logistic Regression
- Support Vector Machine (linear kernel)

Both models use the same preprocessing pipeline, the same train-test split, and evaluation with cross-validation plus test-set metrics.
We also tune the SVM regularization parameter using GridSearchCV.


In [ ]:
# Logistic Regression pipeline
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

lr_cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print('Logistic Regression CV accuracy scores:', lr_cv_scores)
print('Mean CV accuracy:', lr_cv_scores.mean())

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('\nLogistic Regression Accuracy:', accuracy_score(y_test, lr_pred))
print('\nLogistic Regression Confusion Matrix:\n', confusion_matrix(y_test, lr_pred))
print('\nLogistic Regression Classification Report:\n', classification_report(y_test, lr_pred))


In [ ]:
# SVM pipeline with hyperparameter tuning
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='linear'))
])

param_grid = {
    'model__C': [0.1, 1.0, 5.0]
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print('Best SVM parameters:', svm_grid.best_params_)
print('Best SVM CV accuracy:', svm_grid.best_score_)

best_svm = svm_grid.best_estimator_
svm_pred = best_svm.predict(X_test)

print('\nSVM Accuracy:', accuracy_score(y_test, svm_pred))
print('\nSVM Confusion Matrix:\n', confusion_matrix(y_test, svm_pred))
print('\nSVM Classification Report:\n', classification_report(y_test, svm_pred))


In [ ]:
# Visualize confusion matrices for both models
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, ax=axes[0], colorbar=False)
axes[0].set_title('Logistic Regression Confusion Matrix')

ConfusionMatrixDisplay.from_predictions(y_test, svm_pred, ax=axes[1], colorbar=False)
axes[1].set_title('SVM Confusion Matrix')

plt.tight_layout()
plt.show()

results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM'],
    'Test Accuracy': [accuracy_score(y_test, lr_pred), accuracy_score(y_test, svm_pred)],
    'CV Accuracy': [lr_cv_scores.mean(), svm_grid.best_score_]
})

display(results_df.sort_values('Test Accuracy', ascending=False))


## Results Interpretation
- Compare the test accuracy of Logistic Regression and SVM
- Review the confusion matrices to see which model makes fewer classification errors
- Use the classification reports to compare precision, recall, and F1-score
- Prefer the model that balances performance with interpretability and computational practicality


## Conclusion
This notebook provides a complete DS 301 classification workflow using a real public dataset and two class-approved algorithms.
The pipeline includes loading, cleaning, encoding, scaling, EDA, model training, cross-validation, hyperparameter tuning, and evaluation.
The final model selection should be based on the printed metrics after execution.
